In [1]:
import numpy as np
import struct
from array import array
from os.path  import join

%matplotlib inline
import random
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

In [2]:
# MNist reader from:
# https://www.kaggle.com/code/hojjatk/read-mnist-dataset

class MnistDataloader():
    def __init__(self, training_images_filepath,training_labels_filepath,
                 test_images_filepath, test_labels_filepath):
        self.training_images_filepath = training_images_filepath
        self.training_labels_filepath = training_labels_filepath
        self.test_images_filepath = test_images_filepath
        self.test_labels_filepath = test_labels_filepath
    
    def read_images_labels(self, images_filepath, labels_filepath):        
        labels = []
        with open(labels_filepath, 'rb') as file:
            magic, size = struct.unpack(">II", file.read(8))
            if magic != 2049:
                raise ValueError('Magic number mismatch, expected 2049, got {}'.format(magic))
            labels = array("B", file.read())        
        
        with open(images_filepath, 'rb') as file:
            magic, size, rows, cols = struct.unpack(">IIII", file.read(16))
            if magic != 2051:
                raise ValueError('Magic number mismatch, expected 2051, got {}'.format(magic))
            image_data = array("B", file.read())        
        images = []
        for i in range(size):
            images.append(np.zeros((rows, cols)))
        for i in range(size):
            img = np.array(image_data[i * rows * cols:(i + 1) * rows * cols])
            img = img.reshape(28, 28)
            images[i][:] = img            
        
        return images, labels
            
    def load_data(self):
        x_train, y_train = self.read_images_labels(self.training_images_filepath, self.training_labels_filepath)
        x_test, y_test = self.read_images_labels(self.test_images_filepath, self.test_labels_filepath)
        return (x_train, y_train),(x_test, y_test)   

# Helper code to show images

def show_images(images, title_texts):
    cols = 5
    rows = int(len(images)/cols) + 1
    plt.figure(figsize=(30,20))
    index = 1    
    for x in zip(images, title_texts):        
        image = x[0]        
        title_text = x[1]
        plt.subplot(rows, cols, index)        
        plt.imshow(image, cmap=plt.cm.gray)
        if (title_text != ''):
            plt.title(title_text, fontsize = 15);        
        index += 1
        plt.show



In [3]:
#
# Load MINST dataset
#
input_path = 'mnist'
training_images_filepath = join(input_path, 'train-images-idx3-ubyte/train-images-idx3-ubyte')
training_labels_filepath = join(input_path, 'train-labels-idx1-ubyte/train-labels-idx1-ubyte')
test_images_filepath = join(input_path, 't10k-images-idx3-ubyte/t10k-images-idx3-ubyte')
test_labels_filepath = join(input_path, 't10k-labels-idx1-ubyte/t10k-labels-idx1-ubyte')

mnist_dataloader = MnistDataloader(training_images_filepath, training_labels_filepath, test_images_filepath, test_labels_filepath)
(x_train, y_train), (x_test, y_test) = mnist_dataloader.load_data()

# end of code from
# https://www.kaggle.com/code/hojjatk/read-mnist-dataset

In [4]:
# Pytorch dataset and dataloader, convenient wrapper to iterate through data in random order

class MNistDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return torch.tensor(self.x[i], dtype=torch.float32), self.y[i]

dataset = MNistDataset(x_train, y_train)
test_dataset = MNistDataset(x_test, y_test)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=True,
)

In [5]:
class ConvModel(nn.Module):
    def __init__(self, h=32):
        super().__init__()
        self.h = h
        
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn1 = nn.BatchNorm2d(h)
        self.conv2 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn2 = nn.BatchNorm2d(h)
        self.conv3 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn3 = nn.BatchNorm2d(h)

        self.pool1 = nn.AdaptiveMaxPool2d((14, 14))
        self.drop1 = nn.Dropout(0.1)

        self.conv4 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn4 = nn.BatchNorm2d(h)
        self.conv5 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn5 = nn.BatchNorm2d(h)

        self.pool2 = nn.AdaptiveMaxPool2d((7, 7))
        self.drop2 = nn.Dropout(0.1)

        self.conv6 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn6 = nn.BatchNorm2d(h)
        self.conv7 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn7 = nn.BatchNorm2d(h)
        self.conv8 = nn.Conv2d(in_channels=h, out_channels=h, kernel_size=(3, 3), padding="same")
        self.bn8 = nn.BatchNorm2d(h)
        
        self.pool3 = nn.AdaptiveMaxPool2d((1, 1))
        # avg pool: ~4000 below 0.200, 5000 ~0.150
        # max pool: ~1000 below 0.200, 2000 ~0.120, 5000 ~0.060, acc. 98.2%
        # 2 max pools: ~500 below 0.200, 2000 ~0.060, acc. 97.9%, stopping at 2000
        # 3 max pools: similar, acc. 98.1%
        # wider: 98.7%
        self.lin1 = nn.Linear(h, h)
        self.lin2 = nn.Linear(h, 10)

    def forward(self, x):
        h = self.h
        
        # normalize
        x = (x / 128) - 0.5
        x.unsqueeze_(-3)
        # (1, 28, 28)
        x = self.conv1(x)
        # (20, 28, 28)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn1(x)

        x = self.conv2(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn2(x)

        x = self.conv3(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn3(x)

        x = self.pool1(x)
        x = self.drop1(x)

        x = self.conv4(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn4(x)
        
        x = self.conv5(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn5(x)

        x = self.pool2(x)
        x = self.drop2(x)
        
        x = self.conv6(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn6(x)

        
        x = self.conv7(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn7(x)

        x = self.conv8(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.bn8(x)

        x = self.pool3(x)
        x = x.reshape((-1, h, ))
        x = self.lin1(x)
        x = F.leaky_relu(x, negative_slope=0.2)
        x = self.lin2(x)

        return x
        


In [6]:
model = ConvModel()
model_hooks = []

In [7]:
# how many params
sum([p.numel() for p in model.parameters()])

66954

In [8]:
# check that model runs and shapes work
model(torch.tensor(x_train[2], dtype=torch.float32).unsqueeze(0))

tensor([[ 0.4230,  0.7107,  0.0181,  0.2024,  0.1585,  0.1064,  0.6137, -0.3623,
          0.4533,  0.0023]], grad_fn=<AddmmBackward0>)

In [9]:
# training loop

n_batches = 2000
cross_entropy = nn.CrossEntropyLoss()
lr = 0.01

# not doing weight decay for now
# l2_lambda = 0
# decay = []
# for p in model.parameters():
#     if p.requires_grad and p.dim() > 1:
#         decay.append(p)

def test_loss():
    model.eval()
    cml_loss = 0
    cml_items = 0
    for x, y in test_loader:
        n_items = x.shape[0]
        with torch.no_grad():
            y_pred = model(x)
        loss = cross_entropy(y_pred, y)
        cml_loss += loss.item() * n_items
        cml_items += n_items
    model.train()
    print(f"Computed test loss over {cml_items} items")
    return cml_loss / cml_items

for i in range(n_batches):
    x, y = loader.__iter__().__next__()
    y_pred = model(x)
    loss = cross_entropy(y_pred, y)
    if i % 250 == 0 or i == n_batches - 1:
        print(f"Loss at step {i:4} is {loss.item():0.3f}")
        print(f"Test loss at step {i:4} is {test_loss():0.3f}")

    # for p in decay:
    #     loss += l2_lambda * (p*p).mean()

    loss.backward()

    for p in model.parameters():
        p.data -= lr * p.grad
    model.zero_grad()

Loss at step    0 is 2.301
Computed test loss over 10000 items
Test loss at step    0 is 2.309
Loss at step  250 is 0.325
Computed test loss over 10000 items
Test loss at step  250 is 0.241
Loss at step  500 is 0.070
Computed test loss over 10000 items
Test loss at step  500 is 0.100
Loss at step  750 is 0.045
Computed test loss over 10000 items
Test loss at step  750 is 0.067
Loss at step 1000 is 0.067
Computed test loss over 10000 items
Test loss at step 1000 is 0.057
Loss at step 1250 is 0.157
Computed test loss over 10000 items
Test loss at step 1250 is 0.048
Loss at step 1500 is 0.016
Computed test loss over 10000 items
Test loss at step 1500 is 0.046
Loss at step 1750 is 0.052
Computed test loss over 10000 items
Test loss at step 1750 is 0.060
Loss at step 1999 is 0.019
Computed test loss over 10000 items
Test loss at step 1999 is 0.044


In [ ]:
cml_loss = 0
count = 0
for idx in range(200, 215):
    x, y = test_dataset[idx]
    #show_images([x], [y])
    #plt.show()
    model.eval()
    res = model(torch.tensor(x, dtype=torch.float32).unsqueeze(0))
    model.train()
    #print(res)
    loss = cross_entropy(res, torch.tensor([y])).item()
    print(f"{idx:5} -- {loss:0.2f}")
    cml_loss += loss
    count += 1

print(f"Average: {cml_loss/count:0.2f}")




In [10]:
def test_accuracy():
    model.eval()
    cml_correct = 0
    cml_items = 0
    for x, y in test_loader:
        n_items = x.shape[0]
        with torch.no_grad():
            y_pred = model(x)
        predictions = torch.argmax(y_pred, dim=1)
        cml_correct += (y == predictions).sum().item()
        cml_items += n_items
    model.train()
    print(f"Computed test accuracy over {cml_items} items")
    return cml_correct / cml_items

test_accuracy()

Computed test accuracy over 10000 items


0.9873

In [ ]:
x, y = test_dataset[209]
fig, ax = plt.subplots()
ax.imshow(x, cmap=plt.cm.gray)
plt.show()
print(y)
model.eval()
y_pred = model(torch.tensor(x, dtype=torch.float32).unsqueeze(0))
print(y_pred)
print(cross_entropy(y_pred, torch.tensor([y])))

In [ ]:
def print_stats(t, name = ""):
    print(f"Tensor stats: {name}")
    print(f"  Mean: {t.mean():.2e}")
    print(f"  Second moment: {(t * t).mean():.2e}")
    print(f"  Standard dev: {t.std():.2e}")
    print(f"  Min: {torch.min(t):.2e}")
    print(f"      10th quantile: {torch.quantile(t, 0.1):.2e}")
    print(f"      90th quantile: {torch.quantile(t, 0.9):.2e}")
    print(f"  Max: {torch.max(t):.2e}")

def print_param_stats(model):
    for n, p in model.named_parameters():
        if p.dim() > 1:
            print_stats(p, name=n)

In [ ]:

print_param_stats(model)

In [ ]:
model.conv4.weight.shape

In [ ]:
for handle in model_hooks:
    handle.remove()

def print_vals_one_sample(module, x, y):
    y = y.detach()
    print(f"Output of {module}")
    print(f"Mean: {y.mean()}")
    print(f"First few values: {y.flatten()[0:5]}")

layer_stds = []

def print_layer_std(module, x, y):
    y = y.detach()
    fig, ax = plt.subplots()
    ax.imshow(y.std(dim=0)[0], cmap=plt.cm.gray)
    plt.show()
    layer_stds.append(y.std(dim=0)[0])

def print_grad_stats(grad):
    print(f"Gradient stats:")
    print(f"  Mean: {grad.mean():.2e}")
    print(f"  Second moment: {(grad * grad).mean():.2e}")
    print(f"  10th quantile: {torch.quantile(grad, 0.1):.2e}")
    print(f"  90th quantile: {torch.quantile(grad, 0.9):.2e}")

model_hooks.append(model.conv4.weight.register_hook(print_grad_stats))

In [ ]:
layer_stds[0]